# 03 · Features and pipeline: each addition judged on June
**Owner: P2** · Status: `[ ]`

Rules: stateless features live in `src/features.py`; anything learned lives inside the `Pipeline` (`src/models.py`) and is fitted on **May only**. June scores every choice. **July is not loaded here.** Report the features that lost too.

In [1]:
import sys; sys.path.insert(0, "..")          # repo root, so `import src...` works
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.config import TARGET
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 140)

from src.clean import build_datasets
from src.features import FEATURE_SETS, add_features
from src.models import make_pipeline, MODELS
from src.evaluate import metrics

D = build_datasets()
train, val = D["train"], D["val"]            # NB03 never touches D["test"]
ytr, yva = train[TARGET], val[TARGET]
print(len(train), len(val))

133342 133459


## 1. Ablation: feature sets x models (train May, score June)

In [2]:
rows = []
for fs in FEATURE_SETS:
    for name in ["linear", "tree"]:
        pipe = make_pipeline(MODELS[name](), fs).fit(train, ytr)
        rows.append(dict(feature_set=fs, model=name, n_features=pipe[:-1].transform(val.head(5)).shape[1],
                         **{k: round(v, 3) for k, v in metrics(yva, pipe.predict(val)).items() if k != "n"}))
ablation = pd.DataFrame(rows)
ablation.pivot(index="feature_set", columns="model", values=["mae", "rmse"])

mae          rmse       
model       linear   tree linear   tree
feature_set                            
f0_distance  2.979  2.793  4.151  3.981
f1_time      2.819  2.770  3.949  3.949
f2_geo       2.733  2.747  3.850  3.920
f3_airport   2.719  2.747  3.838  3.920
f4_shape     2.590  2.747  3.672  3.920

**Read it:** which addition moved MAE/RMSE, for which model, and by how much? Which did nothing or hurt? Write one line per feature set. (An addition that helps the tree and not the linear model is information.)

## 2. Tune the tree on June (small grid, nothing on July)

In [3]:
from sklearn.tree import DecisionTreeRegressor
fs = "f4_shape"
res = []
for depth in [4, 6, 8, 10, 12]:
    for leaf in [50, 200, 1000]:
        pipe = make_pipeline(DecisionTreeRegressor(max_depth=depth, min_samples_leaf=leaf, random_state=0), fs).fit(train, ytr)
        res.append(dict(depth=depth, leaf=leaf, **{k: round(v, 3) for k, v in metrics(yva, pipe.predict(val)).items() if k != "n"}))
grid = pd.DataFrame(res).sort_values("mae"); grid.head(8)

,depth,leaf,mae,rmse,bias
12,12,50,2.658,3.811,-0.078
9,10,50,2.694,3.854,-0.071
13,12,200,2.701,3.870,-0.082
10,10,200,2.724,3.894,-0.078
6,8,50,2.730,3.896,-0.072
7,8,200,2.747,3.920,-0.073
3,6,50,2.767,3.946,-0.067
4,6,200,2.772,3.951,-0.066


## 3. Ideas to test (each = a new row in the ablation, kept or reported as a loser)
- Peak / overnight windows as indicators (check TLC rules; the surcharges themselves are in `extra`, not in `fare_amount`)
- JFK <-> Manhattan indicator; airport x borough interactions for the linear model
- Zone-pair information: **only inside the pipeline** (`sklearn.preprocessing.TargetEncoder`, fitted on May). A zone-pair mean computed on the full data is a leak.
- Distance splines / piecewise slope for the linear model
- Hour x weekday-vs-weekend for the linear model

Leak check for every idea: *could the booking app know this value at the moment of booking?*

In [4]:
# Template: add a feature to src/features.py (stateless) or a learned step to the pipeline, then
# add a new key to FEATURE_SETS and re-run section 1. Example of a stateless one:
#   d["rush"] = d.hour.between(16, 19) & (d.dow < 5)

## 4. Features that lost / did nothing
| Feature | June MAE change | Verdict |
|---|---|---|
| | | |

## 5. Freeze (Gate 2)
Set `FINAL_FEATURE_SET` and `FINAL_MODELS` in `src/config.py` and write the one-paragraph justification here. After this, only P3 runs NB04.

In [5]:
from src.config import FINAL_FEATURE_SET, FINAL_MODELS
print("frozen:", FINAL_FEATURE_SET, FINAL_MODELS)

frozen: f4_shape ('linear', 'tree')
